In [34]:
%%writefile STAT5243_PROJECT_2.py
from shiny import App, ui, reactive, render
import pandas as pd
import numpy as np
import os
import json
import pyreadr
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# =========================================================
# App paths
# =========================================================
APP_DIR = os.getcwd()
DATA_DIR = os.path.join(APP_DIR, "data")


# =========================================================
# Helper functions
# =========================================================
def read_uploaded_data(fileinfo):
    path = fileinfo["datapath"]
    name = fileinfo["name"].lower()
    ext = name.split(".")[-1]

    if ext in ["csv", "txt"]:
        try:
            df = pd.read_csv(path)
            if df.shape[1] > 1:
                return df
        except Exception:
            pass

        try:
            df = pd.read_csv(path, sep=";")
            if df.shape[1] > 1:
                return df
        except Exception:
            pass

        try:
            df = pd.read_csv(path, sep="\t")
            return df
        except Exception:
            pass

        raise ValueError("Unable to parse the CSV/TXT file. Please check the delimiter.")

    if ext in ["xlsx", "xls"]:
        return pd.read_excel(path)

    if ext == "json":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        return pd.DataFrame(obj)

    if ext == "rds":
        result = pyreadr.read_r(path)
        return list(result.values())[0]

    raise ValueError("Unsupported file format. Please upload a CSV, XLSX, JSON, or RDS file.")


def get_builtin(choice: str) -> pd.DataFrame:
    if choice == "bank-full":
        path = os.path.join(DATA_DIR, "bank-full.csv")
        if not os.path.exists(path):
            raise FileNotFoundError("Built-in Bank Marketing dataset file not found.")
        return pd.read_csv(path, sep=";")

    if choice == "titanic":
        path = os.path.join(DATA_DIR, "titanic.csv")
        if not os.path.exists(path):
            raise FileNotFoundError("Built-in Titanic dataset file not found.")
        return pd.read_csv(path)

    raise ValueError("Unknown built-in dataset.")


def clean_char_na(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            s = df[col].astype("string").str.strip()
            s = s.replace("", pd.NA)
            s = s.replace(
                {
                    "na": pd.NA,
                    "NA": pd.NA,
                    "null": pd.NA,
                    "NULL": pd.NA,
                    "unknown": pd.NA,
                    "Unknown": pd.NA,
                    "?": pd.NA,
                }
            )
            df[col] = s
    return df


def coerce_numeric_if_possible(df: pd.DataFrame, min_numeric_ratio: float = 0.7) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if (
            pd.api.types.is_object_dtype(df[col])
            or pd.api.types.is_string_dtype(df[col])
            or pd.api.types.is_bool_dtype(df[col])
        ):
            parsed = pd.to_numeric(
                df[col].astype(str).str.replace(r"[^0-9\.\-]", "", regex=True),
                errors="coerce",
            )
            ratio = parsed.notna().mean()
            if pd.notna(ratio) and ratio >= min_numeric_ratio:
                df[col] = parsed
    return df


def get_mode(series: pd.Series):
    s = series.dropna()
    if s.empty:
        return np.nan
    return s.mode(dropna=True).iloc[0]


def safe_first_non_missing(series: pd.Series):
    s = series.dropna()
    if s.empty:
        return np.nan
    s = s[s.astype(str).str.strip() != ""]
    if s.empty:
        return np.nan
    return s.iloc[0]


def safe_scale(series: pd.Series):
    if not pd.api.types.is_numeric_dtype(series):
        return series
    if series.dropna().nunique() <= 1:
        return series
    scaler = StandardScaler()
    out = series.copy()
    mask = out.notna()
    out.loc[mask] = scaler.fit_transform(out.loc[mask].values.reshape(-1, 1)).flatten()
    return out


def dataset_info(df: pd.DataFrame) -> pd.DataFrame:
    num_vars = sum(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)
    cat_vars = df.shape[1] - num_vars
    return pd.DataFrame(
        {
            "Metric": [
                "Number of Rows",
                "Number of Columns",
                "Numeric Variables",
                "Categorical Variables",
            ],
            "Value": [df.shape[0], df.shape[1], num_vars, cat_vars],
        }
    )


def make_structure_summary(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "Variable": df.columns,
            "Type": [str(df[c].dtype) for c in df.columns],
            "Missing_Count": [int(df[c].isna().sum()) for c in df.columns],
            "Missing_Percent": [f"{df[c].isna().mean() * 100:.2f}%" for c in df.columns],
        }
    )


def handle_outliers(df: pd.DataFrame, var: str, method: str) -> pd.DataFrame:
    df = df.copy()
    if var not in df.columns or not pd.api.types.is_numeric_dtype(df[var]) or method == "None":
        return df

    x = df[var]
    if x.dropna().empty:
        return df

    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr_val = q3 - q1
    lower = q1 - 1.5 * iqr_val
    upper = q3 + 1.5 * iqr_val

    if method == "Remove by IQR":
        df = df[(x.between(lower, upper)) | (x.isna())]
    elif method == "Cap by IQR":
        df[var] = x.clip(lower=lower, upper=upper)
    elif method == "Log transform":
        if (x.dropna() >= 0).all():
            df[var] = np.log1p(x)

    return df


def clean_data_pipeline(
    raw_df: pd.DataFrame,
    unknown_as_na: bool,
    pdays_as_na: bool,
    coerce_numeric_flag: bool,
    missing_thr: float,
    missing_method: str,
    remove_dup_rows: bool,
    handle_dup_ids: bool,
    outlier_var: str | None,
    outlier_method: str,
    scale_numeric_flag: bool,
    encode_cats: bool,
    cat_cols_selected: list[str],
) -> pd.DataFrame:
    df = raw_df.copy()

    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

    if unknown_as_na:
        df = clean_char_na(df)

    if coerce_numeric_flag:
        df = coerce_numeric_if_possible(df)

    if pdays_as_na and "pdays" in df.columns and pd.api.types.is_numeric_dtype(df["pdays"]):
        df.loc[df["pdays"] == -1, "pdays"] = np.nan

    miss_rate = df.isna().mean()
    keep_cols = miss_rate[miss_rate <= missing_thr].index
    df = df[keep_cols].copy()

    if remove_dup_rows:
        df = df.drop_duplicates()

    lower_map = {c.lower(): c for c in df.columns}
    if handle_dup_ids and "id" in lower_map and "position" in lower_map:
        id_col = lower_map["id"]
        pos_col = lower_map["position"]
        df[id_col] = df.groupby(pos_col)[id_col].transform(
            lambda s: s.fillna(safe_first_non_missing(s))
        )
        df = df.drop_duplicates(subset=[id_col], keep="first")

    if missing_method == "Remove rows with missing values":
        df = df.dropna()

    elif missing_method == "Median impute numeric columns":
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        for col in num_cols:
            if not df[col].dropna().empty:
                df[col] = df[col].fillna(df[col].median())

    elif missing_method == "Mean impute numeric columns":
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        for col in num_cols:
            if not df[col].dropna().empty:
                df[col] = df[col].fillna(df[col].mean())

    elif missing_method == "Mode impute categorical columns":
        cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        for col in cat_cols:
            df[col] = df[col].fillna(get_mode(df[col]))

    elif missing_method == "Do nothing":
        pass

    if outlier_var:
        df = handle_outliers(df, outlier_var, outlier_method)

    if scale_numeric_flag:
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        for col in num_cols:
            df[col] = safe_scale(df[col])

    if encode_cats and cat_cols_selected:
        selected_cols = [c for c in cat_cols_selected if c in df.columns]
        for col in selected_cols:
            if not pd.api.types.is_numeric_dtype(df[col]):
                df[col] = pd.factorize(df[col])[0].astype(float)

    return df


def feature_engineering_pipeline(
    df: pd.DataFrame,
    create_date_parts: bool = False,
    date_var: str | None = None,
    create_age_group: bool = False,
    age_var: str | None = None,
    create_balance_level: bool = False,
    balance_var: str | None = None,
    create_contacted_before: bool = False,
    pdays_var: str | None = None,
    create_log_feature: bool = False,
    log_var: str | None = None,
    create_sqrt_feature: bool = False,
    sqrt_var: str | None = None,
    create_square_feature: bool = False,
    square_var: str | None = None,
    create_ratio_feature: bool = False,
    ratio_num: str | None = None,
    ratio_den: str | None = None,
    create_sum_feature: bool = False,
    sum_var1: str | None = None,
    sum_var2: str | None = None,
    create_diff_feature: bool = False,
    diff_var1: str | None = None,
    diff_var2: str | None = None,
    create_product_feature: bool = False,
    prod_var1: str | None = None,
    prod_var2: str | None = None,
    create_indicator_feature: bool = False,
    indicator_var: str | None = None,
    indicator_threshold: float | None = None,
    create_missing_indicator: bool = False,
    missing_var: str | None = None,
    create_frequency_feature: bool = False,
    freq_var: str | None = None,
    create_group_mean_feature: bool = False,
    group_var: str | None = None,
    target_var: str | None = None,
    create_rank_feature: bool = False,
    rank_var: str | None = None,
    create_text_length: bool = False,
    text_var: str | None = None,
) -> pd.DataFrame:
    df = df.copy()

    if create_date_parts and date_var and date_var in df.columns:
        dt = pd.to_datetime(df[date_var], errors="coerce")
        df[f"{date_var}_year"] = dt.dt.year
        df[f"{date_var}_month"] = dt.dt.month
        df[f"{date_var}_day"] = dt.dt.day
        df[f"{date_var}_weekday"] = dt.dt.day_name()
        df[f"{date_var}_quarter"] = dt.dt.quarter
        df[f"{date_var}_is_weekend"] = dt.dt.weekday.isin([5, 6]).astype("float")

    if create_age_group and age_var and age_var in df.columns and pd.api.types.is_numeric_dtype(df[age_var]):
        df["age_group"] = pd.cut(
            df[age_var],
            bins=[-np.inf, 25, 40, 60, np.inf],
            labels=["Young", "Adult", "Middle", "Senior"],
            include_lowest=True,
        )

    if create_balance_level and balance_var and balance_var in df.columns and pd.api.types.is_numeric_dtype(df[balance_var]):
        x = df[balance_var].dropna()
        if not x.empty and x.nunique() >= 3:
            q1 = x.quantile(0.33)
            q2 = x.quantile(0.67)
            if q1 < q2:
                df["balance_level"] = pd.cut(
                    df[balance_var],
                    bins=[-np.inf, q1, q2, np.inf],
                    labels=["Low", "Medium", "High"],
                    include_lowest=True,
                )

    if create_contacted_before and pdays_var and pdays_var in df.columns and pd.api.types.is_numeric_dtype(df[pdays_var]):
        df["contacted_before"] = np.where(df[pdays_var].fillna(-1) >= 0, "Yes", "No")

    if create_log_feature and log_var and log_var in df.columns and pd.api.types.is_numeric_dtype(df[log_var]):
        x = df[log_var]
        if not x.dropna().empty and (x.dropna() >= 0).all():
            df[f"{log_var}_log"] = np.log1p(x)

    if create_sqrt_feature and sqrt_var and sqrt_var in df.columns and pd.api.types.is_numeric_dtype(df[sqrt_var]):
        x = df[sqrt_var]
        if not x.dropna().empty and (x.dropna() >= 0).all():
            df[f"{sqrt_var}_sqrt"] = np.sqrt(x)

    if create_square_feature and square_var and square_var in df.columns and pd.api.types.is_numeric_dtype(df[square_var]):
        df[f"{square_var}_sq"] = df[square_var] ** 2

    if (
        create_ratio_feature
        and ratio_num and ratio_den
        and ratio_num in df.columns and ratio_den in df.columns
        and pd.api.types.is_numeric_dtype(df[ratio_num])
        and pd.api.types.is_numeric_dtype(df[ratio_den])
    ):
        denom = df[ratio_den].replace(0, np.nan)
        df[f"{ratio_num}_to_{ratio_den}_ratio"] = df[ratio_num] / denom

    if (
        create_sum_feature
        and sum_var1 and sum_var2
        and sum_var1 in df.columns and sum_var2 in df.columns
        and pd.api.types.is_numeric_dtype(df[sum_var1])
        and pd.api.types.is_numeric_dtype(df[sum_var2])
    ):
        df[f"{sum_var1}_plus_{sum_var2}"] = df[sum_var1] + df[sum_var2]

    if (
        create_diff_feature
        and diff_var1 and diff_var2
        and diff_var1 in df.columns and diff_var2 in df.columns
        and pd.api.types.is_numeric_dtype(df[diff_var1])
        and pd.api.types.is_numeric_dtype(df[diff_var2])
    ):
        df[f"{diff_var1}_minus_{diff_var2}"] = df[diff_var1] - df[diff_var2]

    if (
        create_product_feature
        and prod_var1 and prod_var2
        and prod_var1 in df.columns and prod_var2 in df.columns
        and pd.api.types.is_numeric_dtype(df[prod_var1])
        and pd.api.types.is_numeric_dtype(df[prod_var2])
    ):
        df[f"{prod_var1}_x_{prod_var2}"] = df[prod_var1] * df[prod_var2]

    if (
        create_indicator_feature
        and indicator_var
        and indicator_var in df.columns
        and pd.api.types.is_numeric_dtype(df[indicator_var])
        and indicator_threshold is not None
    ):
        safe_thr = str(indicator_threshold).replace(".", "_")
        df[f"{indicator_var}_gt_{safe_thr}"] = (df[indicator_var] > indicator_threshold).astype(int)

    if create_missing_indicator and missing_var and missing_var in df.columns:
        df[f"{missing_var}_missing"] = df[missing_var].isna().astype(int)

    if create_frequency_feature and freq_var and freq_var in df.columns:
        freq_map = df[freq_var].astype(str).value_counts(dropna=False)
        df[f"{freq_var}_freq"] = df[freq_var].astype(str).map(freq_map)

    if (
        create_group_mean_feature
        and group_var and target_var
        and group_var in df.columns and target_var in df.columns
        and pd.api.types.is_numeric_dtype(df[target_var])
    ):
        df[f"{target_var}_mean_by_{group_var}"] = df.groupby(group_var)[target_var].transform("mean")

    if create_rank_feature and rank_var and rank_var in df.columns and pd.api.types.is_numeric_dtype(df[rank_var]):
        df[f"{rank_var}_rank"] = df[rank_var].rank()

    if create_text_length and text_var and text_var in df.columns:
        df[f"{text_var}_length"] = df[text_var].astype("string").str.len()

    return df


# =========================================================
# UI
# =========================================================
app_ui = ui.page_fluid(
    ui.h2("Project 2 - Python Shiny Data Explorer"),

    ui.navset_tab(
        ui.nav_panel(
            "User Guide",
            ui.p(
                "Workflow: Upload your own dataset or select a built-in dataset → "
                "Clean & preprocess → Feature engineering → EDA → Export results."
            ),
            ui.hr(),
            ui.h4("Overview"),
            ui.tags.ul(
                ui.tags.li("Loading Datasets: Supports various format uploads and provides built-in datasets (e.g., Bank Marketing and Titanic)."),
                ui.tags.li("Data Cleaning & Preprocessing: Handle inconsistencies, missing values, duplicates, and apply optional scaling and encoding."),
                ui.tags.li("Feature Engineering: Create new features such as age_group, balance_level, contacted_before, and other derived variables."),
                ui.tags.li("EDA (Exploratory Data Analysis): Interactive filtering, visualization, and dynamic statistical summaries."),
                ui.tags.li("Export: Download cleaned and transformed datasets for reproducibility and reporting."),
            ),
        ),

        ui.nav_panel(
            "1) Data Upload & Preview",
            ui.layout_sidebar(
                ui.sidebar(
                    ui.h4("Load Dataset"),
                    ui.input_radio_buttons(
                        "data_source",
                        "Choose data source:",
                        {"upload": "Upload File", "builtin": "Built-in Dataset"},
                        selected="upload",
                    ),
                    ui.panel_conditional(
                        "input.data_source === 'upload'",
                        ui.input_file(
                            "upload",
                            "Upload dataset (CSV / XLSX / JSON / RDS)",
                            accept=[".csv", ".txt", ".xlsx", ".xls", ".json", ".rds"],
                        ),
                    ),
                    ui.panel_conditional(
                        "input.data_source === 'builtin'",
                        ui.input_select(
                            "builtin_choice",
                            "Select a built-in dataset:",
                            {"bank-full": "Bank Marketing", "titanic": "Titanic"},
                        ),
                    ),
                    ui.input_action_button("load_btn", "Load / Reset"),
                    ui.hr(),
                    ui.input_checkbox("strings_to_factor", "Convert character columns to categorical", True),
                    ui.input_checkbox("show_preview", "Show data preview", True),
                    ui.hr(),
                    ui.p("Click 'Load / Reset' to refresh the dataset and update the downstream tabs."),
                    width=320,
                ),

                ui.h4("Dataset Overview"),
                ui.output_data_frame("data_info"),
                ui.hr(),

                ui.panel_conditional(
                    "input.show_preview === true",
                    ui.h4("Raw Data Preview"),
                    ui.output_data_frame("raw_preview"),
                    ui.hr(),
                ),

                ui.h4("Variable Structure & Raw Missing Summary"),
                ui.p("Placeholder values such as 'unknown' are treated during the cleaning stage."),
                ui.output_data_frame("structure_summary"),
                ui.hr(),

                ui.h4("Raw Summary"),
                ui.output_text_verbatim("raw_summary"),
            ),
        ),

        ui.nav_panel(
            "2) Cleaning & Preprocessing",
            ui.layout_sidebar(
                ui.sidebar(
                    ui.h4("Inconsistency Handling"),
                    ui.input_checkbox("unknown_as_na", "Treat 'unknown' / '?' / 'NULL' as missing", True),
                    ui.input_checkbox("pdays_as_na", "Treat pdays = -1 as missing", True),
                    ui.input_checkbox("coerce_numeric", "Convert numeric-like text columns to numeric", True),

                    ui.hr(),
                    ui.h4("Missing Data"),
                    ui.input_slider("missing_thr", "Drop columns with missing rate >", 0.50, 0.99, 0.95, step=0.01),
                    ui.input_select(
                        "missing_method",
                        "Missing value strategy:",
                        {
                            "Do nothing": "Do nothing",
                            "Remove rows with missing values": "Remove rows with missing values",
                            "Median impute numeric columns": "Median impute numeric columns",
                            "Mean impute numeric columns": "Mean impute numeric columns",
                            "Mode impute categorical columns": "Mode impute categorical columns",
                        },
                    ),

                    ui.hr(),
                    ui.h4("Duplicate / ID Handling"),
                    ui.input_checkbox("remove_dup_rows", "Remove duplicate rows", False),
                    ui.input_checkbox("handle_dup_ids", "Handle duplicate/missing id (if id & position exist)", True),

                    ui.hr(),
                    ui.h4("Outlier Handling"),
                    ui.output_ui("outlier_var_ui"),
                    ui.input_select(
                        "outlier_method",
                        "Outlier strategy:",
                        {
                            "None": "None",
                            "Remove by IQR": "Remove by IQR",
                            "Cap by IQR": "Cap by IQR",
                            "Log transform": "Log transform",
                        },
                    ),

                    ui.hr(),
                    ui.h4("Transformations"),
                    ui.input_checkbox("scale_numeric", "Scale numeric features (z-score)", False),
                    ui.input_checkbox("encode_cats", "Encode categorical columns (label encode)", False),
                    ui.output_ui("cat_cols_ui"),

                    ui.hr(),
                    ui.input_action_button("clean_btn", "Apply Cleaning"),
                    width=320,
                ),

                ui.h4("Before vs After Summary"),
                ui.output_data_frame("cleaning_summary"),
                ui.hr(),

                ui.h4("Cleaned Preview"),
                ui.output_data_frame("clean_preview"),
                ui.hr(),

                ui.h4("Missing Summary After Cleaning"),
                ui.output_data_frame("cleaned_missing_summary"),
                ui.hr(),

                ui.h4("Outlier Preview"),
                ui.output_plot("outlier_plot", height="300px"),
                ui.hr(),

                ui.h4("Cleaning Summary"),
                ui.output_text_verbatim("clean_summary"),
            ),
        ),

        ui.nav_panel(
            "3) Feature Engineering",
            ui.layout_sidebar(
                ui.sidebar(
                    ui.h4("Create New Features"),

                    ui.input_checkbox("create_date_parts", "Extract year / month / day / weekday / quarter", False),
                    ui.output_ui("date_var_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_age_group", "Create age_group", False),
                    ui.output_ui("age_var_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_balance_level", "Create balance_level", False),
                    ui.output_ui("balance_var_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_contacted_before", "Create contacted_before", False),
                    ui.output_ui("pdays_var_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_log_feature", "Create log-transformed feature", False),
                    ui.output_ui("log_var_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_sqrt_feature", "Create square-root feature", False),
                    ui.output_ui("sqrt_var_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_square_feature", "Create square feature", False),
                    ui.output_ui("square_var_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_ratio_feature", "Create ratio feature", False),
                    ui.output_ui("ratio_vars_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_sum_feature", "Create sum feature", False),
                    ui.output_ui("sum_vars_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_diff_feature", "Create difference feature", False),
                    ui.output_ui("diff_vars_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_product_feature", "Create interaction/product feature", False),
                    ui.output_ui("prod_vars_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_indicator_feature", "Create threshold indicator", False),
                    ui.output_ui("indicator_var_ui"),
                    ui.output_ui("indicator_threshold_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_missing_indicator", "Create missing-value indicator", False),
                    ui.output_ui("missing_var_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_frequency_feature", "Create frequency feature", False),
                    ui.output_ui("freq_var_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_group_mean_feature", "Create group mean feature", False),
                    ui.output_ui("group_var_ui"),
                    ui.output_ui("target_var_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_rank_feature", "Create rank feature", False),
                    ui.output_ui("rank_var_ui"),

                    ui.hr(),
                    ui.input_checkbox("create_text_length", "Create text length feature", False),
                    ui.output_ui("text_var_ui"),

                    ui.hr(),
                    ui.input_action_button("feature_btn", "Apply Feature Engineering"),
                    width=340,
                ),

                ui.h4("Feature Engineering Summary"),
                ui.output_data_frame("feature_summary"),
                ui.hr(),

                ui.h4("Engineered Data Preview"),
                ui.output_data_frame("engineered_preview"),
                ui.hr(),

                ui.h4("New Columns Added"),
                ui.output_text_verbatim("new_columns_text"),
                ui.hr(),

                ui.h4("Feature Impact Preview"),
                ui.output_plot("feature_plot", height="300px"),
            ),
        ),

        ui.nav_panel(
            "4) Exploratory Data Analysis (EDA)",
            ui.layout_sidebar(
                ui.sidebar(
                    ui.h4("Filter Dataset"),

                    ui.output_ui("eda_filter_cat_var_ui"),
                    ui.output_ui("eda_filter_cat_value_ui"),

                    ui.hr(),
                    ui.output_ui("eda_filter_num_var_ui"),
                    ui.output_ui("eda_filter_num_range_ui"),

                    ui.hr(),
                    ui.h4("Visualization Options"),
                    ui.input_select(
                        "eda_plot_type",
                        "Select plot type:",
                        {
                            "Histogram": "Histogram",
                            "Boxplot": "Boxplot",
                            "Scatter Plot": "Scatter Plot",
                            "Bar Chart": "Bar Chart",
                        },
                    ),

                    ui.output_ui("eda_x_var_ui"),
                    ui.output_ui("eda_y_var_ui"),
                    ui.output_ui("eda_group_var_ui"),

                    ui.hr(),
                    ui.input_action_button("eda_btn", "Apply EDA"),
                    width=320,
                ),

                ui.h4("Filtered Data Preview"),
                ui.output_data_frame("eda_preview"),
                ui.hr(),

                ui.h4("Summary Statistics"),
                ui.output_data_frame("eda_summary"),
                ui.hr(),

                ui.h4("Missing Summary"),
                ui.output_data_frame("eda_missing_summary"),
                ui.hr(),

                ui.h4("Correlation Matrix"),
                ui.output_data_frame("eda_corr_table"),
                ui.hr(),

                ui.h4("Interactive Visualization"),
                ui.output_plot("eda_plot", height="400px"),
            ),
        ),
    ),
)


# =========================================================
# Server
# =========================================================
def server(input, output, session):
    @reactive.calc
    @reactive.event(input.load_btn)
    def raw_data():
        df = None

        if input.data_source() == "upload":
            files = input.upload()
            if files is None or len(files) == 0:
                raise ValueError("Please upload a file.")
            df = read_uploaded_data(files[0])

        elif input.data_source() == "builtin":
            df = get_builtin(input.builtin_choice())

        for col in df.columns:
            if (
                pd.api.types.is_object_dtype(df[col])
                or pd.api.types.is_string_dtype(df[col])
                or pd.api.types.is_categorical_dtype(df[col])
            ):
                df[col] = df[col].astype("string").replace(r"^\s*$", np.nan, regex=True)

        if input.strings_to_factor():
            for col in df.columns:
                if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
                    df[col] = df[col].astype("category")

        return df

    @render.data_frame
    def data_info():
        return render.DataGrid(dataset_info(raw_data()), width="100%")

    @render.data_frame
    def raw_preview():
        return render.DataGrid(raw_data().head(10), width="100%")

    @render.data_frame
    def structure_summary():
        return render.DataGrid(make_structure_summary(raw_data()), width="100%")

    @render.text
    def raw_summary():
        df = raw_data()
        return f"Rows: {df.shape[0]}\nColumns: {df.shape[1]}\n\n{df.describe(include='all').to_string()}"

    @render.ui
    def cat_cols_ui():
        df = raw_data()
        cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        if len(cat_cols) == 0:
            return ui.p("No categorical columns available for encoding.")
        return ui.input_checkbox_group(
            "cat_cols_selected",
            "Categorical columns to encode:",
            choices={c: c for c in cat_cols},
            selected=cat_cols[: min(5, len(cat_cols))],
        )

    @render.ui
    def outlier_var_ui():
        df = raw_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(num_cols) == 0:
            return ui.p("No numeric variables available for outlier handling.")
        return ui.input_select("outlier_var", "Select numeric variable:", {c: c for c in num_cols})

    @reactive.calc
    @reactive.event(input.clean_btn)
    def clean_data():
        df = raw_data().copy()
        outlier_var = input.outlier_var() if len([c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]) > 0 else None
        cat_cols_selected = input.cat_cols_selected() if input.cat_cols_selected() is not None else []

        return clean_data_pipeline(
            raw_df=df,
            unknown_as_na=input.unknown_as_na(),
            pdays_as_na=input.pdays_as_na(),
            coerce_numeric_flag=input.coerce_numeric(),
            missing_thr=input.missing_thr(),
            missing_method=input.missing_method(),
            remove_dup_rows=input.remove_dup_rows(),
            handle_dup_ids=input.handle_dup_ids(),
            outlier_var=outlier_var,
            outlier_method=input.outlier_method(),
            scale_numeric_flag=input.scale_numeric(),
            encode_cats=input.encode_cats(),
            cat_cols_selected=cat_cols_selected,
        )

    @render.data_frame
    def cleaning_summary():
        before_df = raw_data()
        after_df = clean_data()
        summary_df = pd.DataFrame(
            {
                "Metric": ["Rows", "Columns", "Total Missing Values", "Duplicate Rows"],
                "Before": [
                    before_df.shape[0],
                    before_df.shape[1],
                    int(before_df.isna().sum().sum()),
                    int(before_df.duplicated().sum()),
                ],
                "After": [
                    after_df.shape[0],
                    after_df.shape[1],
                    int(after_df.isna().sum().sum()),
                    int(after_df.duplicated().sum()),
                ],
            }
        )
        return render.DataGrid(summary_df, width="100%")

    @render.data_frame
    def clean_preview():
        return render.DataGrid(clean_data().head(10), width="100%")

    @render.data_frame
    def cleaned_missing_summary():
        return render.DataGrid(make_structure_summary(clean_data()), width="100%")

    @render.plot
    def outlier_plot():
        before_df = raw_data()
        after_df = clean_data()
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))

        num_cols = [c for c in before_df.columns if pd.api.types.is_numeric_dtype(before_df[c])]
        if len(num_cols) == 0:
            axes[0].text(0.5, 0.5, "No numeric variables available.", ha="center", va="center")
            axes[0].set_axis_off()
            axes[1].set_axis_off()
            return fig

        var = input.outlier_var()
        if var not in before_df.columns or not pd.api.types.is_numeric_dtype(before_df[var]):
            axes[0].text(0.5, 0.5, "Please select a valid numeric variable.", ha="center", va="center")
            axes[0].set_axis_off()
            axes[1].set_axis_off()
            return fig

        axes[0].boxplot(before_df[var].dropna())
        axes[0].set_title("Before Cleaning")
        axes[0].set_ylabel(var)

        if var in after_df.columns and pd.api.types.is_numeric_dtype(after_df[var]):
            axes[1].boxplot(after_df[var].dropna())
            axes[1].set_title("After Cleaning")
            axes[1].set_ylabel(var)
        else:
            axes[1].text(0.5, 0.5, "Selected variable not available after cleaning.", ha="center", va="center")
            axes[1].set_axis_off()

        plt.tight_layout()
        return fig

    @render.text
    def clean_summary():
        before_df = raw_data()
        after_df = clean_data()
        return (
            "Cleaning completed successfully.\n\n"
            f"Rows before: {before_df.shape[0]}\n"
            f"Rows after:  {after_df.shape[0]}\n"
            f"Columns before: {before_df.shape[1]}\n"
            f"Columns after:  {after_df.shape[1]}\n"
            f"Total missing values before: {int(before_df.isna().sum().sum())}\n"
            f"Total missing values after:  {int(after_df.isna().sum().sum())}\n"
            f"Duplicate rows before: {int(before_df.duplicated().sum())}\n"
            f"Duplicate rows after:  {int(after_df.duplicated().sum())}"
        )

    @render.ui
    def date_var_ui():
        df = clean_data()
        cols = df.columns.tolist()
        if not cols:
            return ui.p("No columns available.")
        return ui.input_select("date_var", "Select date variable:", {c: c for c in cols})

    @render.ui
    def age_var_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            return ui.p("No numeric columns available.")
        return ui.input_select("age_var", "Select age variable:", {c: c for c in num_cols}, selected="age" if "age" in num_cols else num_cols[0])

    @render.ui
    def balance_var_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            return ui.p("No numeric columns available.")
        return ui.input_select("balance_var", "Select balance variable:", {c: c for c in num_cols}, selected="balance" if "balance" in num_cols else num_cols[0])

    @render.ui
    def pdays_var_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            return ui.p("No numeric columns available.")
        return ui.input_select("pdays_var", "Select pdays-like variable:", {c: c for c in num_cols}, selected="pdays" if "pdays" in num_cols else num_cols[0])

    @render.ui
    def log_var_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            return ui.p("No numeric columns available.")
        return ui.input_select("log_var", "Select variable for log transform:", {c: c for c in num_cols})

    @render.ui
    def sqrt_var_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            return ui.p("No numeric columns available.")
        return ui.input_select("sqrt_var", "Select variable:", {c: c for c in num_cols})

    @render.ui
    def square_var_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            return ui.p("No numeric columns available.")
        return ui.input_select("square_var", "Select variable:", {c: c for c in num_cols})

    @render.ui
    def ratio_vars_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(num_cols) < 2:
            return ui.p("Need at least two numeric columns.")
        return ui.TagList(
            ui.input_select("ratio_num", "Numerator:", {c: c for c in num_cols}, selected=num_cols[0]),
            ui.input_select("ratio_den", "Denominator:", {c: c for c in num_cols}, selected=num_cols[1]),
        )

    @render.ui
    def sum_vars_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(num_cols) < 2:
            return ui.p("Need at least two numeric columns.")
        return ui.TagList(
            ui.input_select("sum_var1", "First variable:", {c: c for c in num_cols}, selected=num_cols[0]),
            ui.input_select("sum_var2", "Second variable:", {c: c for c in num_cols}, selected=num_cols[1]),
        )

    @render.ui
    def diff_vars_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(num_cols) < 2:
            return ui.p("Need at least two numeric columns.")
        return ui.TagList(
            ui.input_select("diff_var1", "First variable:", {c: c for c in num_cols}, selected=num_cols[0]),
            ui.input_select("diff_var2", "Second variable:", {c: c for c in num_cols}, selected=num_cols[1]),
        )

    @render.ui
    def prod_vars_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(num_cols) < 2:
            return ui.p("Need at least two numeric columns.")
        return ui.TagList(
            ui.input_select("prod_var1", "First variable:", {c: c for c in num_cols}, selected=num_cols[0]),
            ui.input_select("prod_var2", "Second variable:", {c: c for c in num_cols}, selected=num_cols[1]),
        )

    @render.ui
    def indicator_var_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            return ui.p("No numeric columns available.")
        return ui.input_select("indicator_var", "Select variable:", {c: c for c in num_cols})

    @render.ui
    def indicator_threshold_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            return ui.p("No numeric columns available.")
        default_var = input.indicator_var() if input.indicator_var() in df.columns else num_cols[0]
        s = df[default_var].dropna()
        if s.empty:
            return ui.p("Selected variable has no valid values.")
        return ui.input_numeric("indicator_threshold", "Threshold:", value=float(s.median()))

    @render.ui
    def missing_var_ui():
        df = clean_data()
        cols = df.columns.tolist()
        if not cols:
            return ui.p("No columns available.")
        return ui.input_select("missing_var", "Select variable:", {c: c for c in cols})

    @render.ui
    def freq_var_ui():
        df = clean_data()
        cols = df.columns.tolist()
        if not cols:
            return ui.p("No columns available.")
        return ui.input_select("freq_var", "Select categorical/text variable:", {c: c for c in cols})

    @render.ui
    def group_var_ui():
        df = clean_data()
        cols = df.columns.tolist()
        if not cols:
            return ui.p("No columns available.")
        return ui.input_select("group_var", "Group variable:", {c: c for c in cols})

    @render.ui
    def target_var_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            return ui.p("No numeric columns available.")
        return ui.input_select("target_var", "Target numeric variable:", {c: c for c in num_cols})

    @render.ui
    def rank_var_ui():
        df = clean_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            return ui.p("No numeric columns available.")
        return ui.input_select("rank_var", "Select variable:", {c: c for c in num_cols})

    @render.ui
    def text_var_ui():
        df = clean_data()
        cols = df.columns.tolist()
        if not cols:
            return ui.p("No columns available.")
        return ui.input_select("text_var", "Select text variable:", {c: c for c in cols})

    @reactive.calc
    @reactive.event(input.feature_btn)
    def engineered_data():
        df = clean_data().copy()

        return feature_engineering_pipeline(
            df=df,
            create_date_parts=input.create_date_parts(),
            date_var=input.date_var() if input.create_date_parts() else None,
            create_age_group=input.create_age_group(),
            age_var=input.age_var() if input.create_age_group() else None,
            create_balance_level=input.create_balance_level(),
            balance_var=input.balance_var() if input.create_balance_level() else None,
            create_contacted_before=input.create_contacted_before(),
            pdays_var=input.pdays_var() if input.create_contacted_before() else None,
            create_log_feature=input.create_log_feature(),
            log_var=input.log_var() if input.create_log_feature() else None,
            create_sqrt_feature=input.create_sqrt_feature(),
            sqrt_var=input.sqrt_var() if input.create_sqrt_feature() else None,
            create_square_feature=input.create_square_feature(),
            square_var=input.square_var() if input.create_square_feature() else None,
            create_ratio_feature=input.create_ratio_feature(),
            ratio_num=input.ratio_num() if input.create_ratio_feature() else None,
            ratio_den=input.ratio_den() if input.create_ratio_feature() else None,
            create_sum_feature=input.create_sum_feature(),
            sum_var1=input.sum_var1() if input.create_sum_feature() else None,
            sum_var2=input.sum_var2() if input.create_sum_feature() else None,
            create_diff_feature=input.create_diff_feature(),
            diff_var1=input.diff_var1() if input.create_diff_feature() else None,
            diff_var2=input.diff_var2() if input.create_diff_feature() else None,
            create_product_feature=input.create_product_feature(),
            prod_var1=input.prod_var1() if input.create_product_feature() else None,
            prod_var2=input.prod_var2() if input.create_product_feature() else None,
            create_indicator_feature=input.create_indicator_feature(),
            indicator_var=input.indicator_var() if input.create_indicator_feature() else None,
            indicator_threshold=input.indicator_threshold() if input.create_indicator_feature() else None,
            create_missing_indicator=input.create_missing_indicator(),
            missing_var=input.missing_var() if input.create_missing_indicator() else None,
            create_frequency_feature=input.create_frequency_feature(),
            freq_var=input.freq_var() if input.create_frequency_feature() else None,
            create_group_mean_feature=input.create_group_mean_feature(),
            group_var=input.group_var() if input.create_group_mean_feature() else None,
            target_var=input.target_var() if input.create_group_mean_feature() else None,
            create_rank_feature=input.create_rank_feature(),
            rank_var=input.rank_var() if input.create_rank_feature() else None,
            create_text_length=input.create_text_length(),
            text_var=input.text_var() if input.create_text_length() else None,
        )

    @render.data_frame
    def feature_summary():
        before_df = clean_data()
        after_df = engineered_data()
        summary_df = pd.DataFrame(
            {
                "Metric": ["Rows", "Columns"],
                "Before": [before_df.shape[0], before_df.shape[1]],
                "After": [after_df.shape[0], after_df.shape[1]],
            }
        )
        return render.DataGrid(summary_df, width="100%")

    @render.data_frame
    def engineered_preview():
        return render.DataGrid(engineered_data().head(10), width="100%")

    @render.text
    def new_columns_text():
        before_cols = set(clean_data().columns)
        after_cols = set(engineered_data().columns)
        new_cols = sorted(list(after_cols - before_cols))
        if not new_cols:
            return "No new columns were added."
        return "New columns added:\n- " + "\n- ".join(new_cols)

    @render.plot
    def feature_plot():
        before_df = clean_data()
        after_df = engineered_data()

        fig, ax = plt.subplots(figsize=(6, 4))
        new_cols = [c for c in after_df.columns if c not in before_df.columns]

        if not new_cols:
            ax.text(0.5, 0.5, "No engineered feature to display.", ha="center", va="center")
            ax.set_axis_off()
            return fig

        first_new = new_cols[0]

        if pd.api.types.is_numeric_dtype(after_df[first_new]):
            ax.hist(after_df[first_new].dropna(), bins=20)
            ax.set_title(f"Distribution of {first_new}")
            ax.set_xlabel(first_new)
            ax.set_ylabel("Count")
        else:
            after_df[first_new].astype(str).value_counts(dropna=False).head(20).plot(kind="bar", ax=ax)
            ax.set_title(f"Counts of {first_new}")
            ax.set_xlabel(first_new)
            ax.set_ylabel("Count")

        plt.tight_layout()
        return fig

    # =========================
    # EDA UI
    # =========================
    @render.ui
    def eda_filter_cat_var_ui():
        df = engineered_data()
        cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        choices = {"None": "None", **{c: c for c in cat_cols}}
        return ui.input_select(
            "eda_filter_cat_var",
            "Categorical filter variable:",
            choices,
            selected="None",
        )

    @render.ui
    def eda_filter_cat_value_ui():
        df = engineered_data()
        var = input.eda_filter_cat_var()

        if var is None or var == "None" or var not in df.columns:
            return ui.p("No categorical value filter selected.")

        values = sorted(df[var].dropna().astype(str).unique().tolist())
        choices = {"All": "All", **{v: v for v in values}}

        return ui.input_select(
            "eda_filter_cat_value",
            "Categorical value:",
            choices,
            selected="All",
        )

    @render.ui
    def eda_filter_num_var_ui():
        df = engineered_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        choices = {"None": "None", **{c: c for c in num_cols}}
        return ui.input_select(
            "eda_filter_num_var",
            "Numeric filter variable:",
            choices,
            selected="None",
        )

    @render.ui
    def eda_filter_num_range_ui():
        df = engineered_data()
        var = input.eda_filter_num_var()

        if var is None or var == "None" or var not in df.columns:
            return ui.p("No numeric range filter selected.")

        s = df[var].dropna()
        if s.empty:
            return ui.p("Selected numeric column has no valid values.")

        return ui.input_slider(
            "eda_filter_num_range",
            f"Range for {var}:",
            min=float(s.min()),
            max=float(s.max()),
            value=(float(s.min()), float(s.max())),
        )

    @render.ui
    def eda_x_var_ui():
        df = engineered_data()
        cols = df.columns.tolist()
        if not cols:
            return ui.p("No variables available.")
        return ui.input_select(
            "eda_x_var",
            "X variable:",
            {c: c for c in cols},
            selected=cols[0],
        )

    @render.ui
    def eda_y_var_ui():
        df = engineered_data()
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(num_cols) == 0:
            return ui.p("No numeric Y variable available.")
        return ui.input_select(
            "eda_y_var",
            "Y variable (for scatter/boxplot):",
            {c: c for c in num_cols},
            selected=num_cols[0],
        )

    @render.ui
    def eda_group_var_ui():
        df = engineered_data()
        cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        choices = {"None": "None", **{c: c for c in cat_cols}}
        return ui.input_select(
            "eda_group_var",
            "Group / color variable:",
            choices,
            selected="None",
        )

    # =========================
    # EDA filtered data
    # =========================
    @reactive.calc
    @reactive.event(input.eda_btn)
    def eda_data():
        df = engineered_data().copy()

        cat_var = input.eda_filter_cat_var()
        cat_val = input.eda_filter_cat_value()

        if cat_var and cat_var != "None" and cat_var in df.columns and cat_val != "All":
            df = df[df[cat_var].astype(str) == str(cat_val)]

        num_var = input.eda_filter_num_var()
        if num_var and num_var != "None" and num_var in df.columns:
            rng = input.eda_filter_num_range()
            if rng is not None:
                df = df[df[num_var].between(rng[0], rng[1], inclusive="both")]

        return df

    # =========================
    # EDA outputs
    # =========================
    @render.data_frame
    def eda_preview():
        return render.DataGrid(eda_data().head(15), width="100%")

    @render.data_frame
    def eda_summary():
        df = eda_data()
        num_df = df.select_dtypes(include=[np.number])

        if num_df.shape[1] == 0:
            return render.DataGrid(
                pd.DataFrame({"Message": ["No numeric columns available."]}),
                width="100%",
            )

        summary = num_df.describe().T.reset_index().rename(columns={"index": "Variable"})
        return render.DataGrid(summary, width="100%")

    @render.data_frame
    def eda_missing_summary():
        df = eda_data()
        missing_df = pd.DataFrame(
            {
                "Variable": df.columns,
                "Missing_Count": [int(df[c].isna().sum()) for c in df.columns],
                "Missing_Percent": [f"{df[c].isna().mean() * 100:.2f}%" for c in df.columns],
            }
        )
        return render.DataGrid(missing_df, width="100%")

    @render.data_frame
    def eda_corr_table():
        df = eda_data()
        num_df = df.select_dtypes(include=[np.number])

        if num_df.shape[1] < 2:
            return render.DataGrid(
                pd.DataFrame({"Message": ["Need at least two numeric columns for correlation."]}),
                width="100%",
            )

        corr = num_df.corr().round(3)
        corr_df = corr.reset_index().rename(columns={"index": "Variable"})
        return render.DataGrid(corr_df, width="100%")

    @render.plot
    def eda_plot():
        df = eda_data()
        fig, ax = plt.subplots(figsize=(8, 5))

        if df.empty:
            ax.text(0.5, 0.5, "No data available after filtering.", ha="center", va="center")
            ax.set_axis_off()
            return fig

        plot_type = input.eda_plot_type()
        x_var = input.eda_x_var()
        y_var = input.eda_y_var()
        group_var = input.eda_group_var()

        if plot_type == "Histogram":
            if x_var not in df.columns or not pd.api.types.is_numeric_dtype(df[x_var]):
                ax.text(0.5, 0.5, "Histogram requires a numeric X variable.", ha="center", va="center")
                ax.set_axis_off()
                return fig

            ax.hist(df[x_var].dropna(), bins=20)
            ax.set_title(f"Histogram of {x_var}")
            ax.set_xlabel(x_var)
            ax.set_ylabel("Count")

        elif plot_type == "Boxplot":
            if y_var not in df.columns or not pd.api.types.is_numeric_dtype(df[y_var]):
                ax.text(0.5, 0.5, "Boxplot requires numeric Y variable.", ha="center", va="center")
                ax.set_axis_off()
                return fig

            if group_var != "None" and group_var in df.columns and not pd.api.types.is_numeric_dtype(df[group_var]):
                groups = []
                labels = []
                for g in df[group_var].dropna().astype(str).unique():
                    vals = df.loc[df[group_var].astype(str) == g, y_var].dropna()
                    if len(vals) > 0:
                        groups.append(vals)
                        labels.append(g)

                if groups:
                    ax.boxplot(groups, tick_labels=labels)
                    ax.set_title(f"Boxplot of {y_var} by {group_var}")
                    ax.set_xlabel(group_var)
                    ax.set_ylabel(y_var)
                else:
                    ax.text(0.5, 0.5, "No valid grouped data for boxplot.", ha="center", va="center")
                    ax.set_axis_off()
            else:
                ax.boxplot(df[y_var].dropna())
                ax.set_title(f"Boxplot of {y_var}")
                ax.set_ylabel(y_var)

        elif plot_type == "Scatter Plot":
            if (
                x_var not in df.columns
                or y_var not in df.columns
                or not pd.api.types.is_numeric_dtype(df[x_var])
                or not pd.api.types.is_numeric_dtype(df[y_var])
            ):
                ax.text(0.5, 0.5, "Scatter plot requires numeric X and Y variables.", ha="center", va="center")
                ax.set_axis_off()
                return fig

            if group_var != "None" and group_var in df.columns and not pd.api.types.is_numeric_dtype(df[group_var]):
                for g in df[group_var].dropna().astype(str).unique():
                    part = df[df[group_var].astype(str) == g]
                    ax.scatter(part[x_var], part[y_var], label=g, alpha=0.7)
                ax.legend(title=group_var)
            else:
                ax.scatter(df[x_var], df[y_var], alpha=0.7)

            ax.set_title(f"{y_var} vs {x_var}")
            ax.set_xlabel(x_var)
            ax.set_ylabel(y_var)

        elif plot_type == "Bar Chart":
            if x_var not in df.columns:
                ax.text(0.5, 0.5, "Please select a valid X variable.", ha="center", va="center")
                ax.set_axis_off()
                return fig

            counts = df[x_var].astype(str).value_counts(dropna=False).head(20)
            ax.bar(counts.index, counts.values)
            ax.set_title(f"Bar Chart of {x_var}")
            ax.set_xlabel(x_var)
            ax.set_ylabel("Count")
            ax.tick_params(axis="x", rotation=45)

        plt.tight_layout()
        return fig


app = App(app_ui, server)

Overwriting STAT5243_PROJECT_2.py
